# SimCSE and teacher–student comparison systems on HateXplain

This notebook runs the three external comparison systems of the paper *A Quality-Aware Self-Training Framework for Semi-Supervised Hate Speech Detection* on HateXplain. Each system is evaluated once on the same test partition as the proposed framework. The results are the HateXplain row of Table 6.

| System | Description |
|---|---|
| SimCSE full | Frozen supervised SimCSE embeddings (`princeton-nlp/sup-simcse-bert-base-uncased`), a five-member MLP ensemble, and three rounds of ensemble- and graph-supported pseudo-label selection |
| SimCSE confidence | The same embeddings and ensemble with one round of confidence-only selection: threshold 0.95, hard targets, weight 0.5 |
| Ludwig-style teacher–student | A teacher fine-tuned on L labels the unlabeled pool; predictions with confidence of at least 0.80 become hard pseudo-labels with unit weight, and a student is trained on the labeled and pseudo-labeled examples |

**Encoder of the teacher–student system.** Ludwig et al. use XLM-RoBERTa. This implementation uses `bert-base-uncased`, so the baseline shares its encoder with the proposed framework. It follows their threshold-based selection and is not a reproduction of their reported results.

**Labeled data.** The SimCSE systems and the teacher–student student are fitted on L ∪ V. The proposed framework trains on the fitting subset of L, so these systems see more labeled examples. They place the framework's scores in context and are not an equal-supervision comparison.

**Results of the stored run.** Test macro-F1: SimCSE full 0.7213, SimCSE confidence 0.7282, Ludwig-style teacher–student 0.7743.

**Requirements.** A GPU and internet access for the pretrained models and the dataset. The stored run used one NVIDIA Tesla T4. Completed systems are cached in the output directory; set `RESUME_FROM` to continue from a saved output folder.

## Data

HateXplain is downloaded from its official repository at a pinned commit and verified by checksum.

- Labels are assigned by majority vote among the three annotators. The 919 posts without a majority label are excluded.
- `hatespeech` and `offensive` form the positive class and `normal` the negative class.
- Posts whose normalized text appears with conflicting binary labels are removed, and repeated normalized texts are collapsed to one post.
- The larger class is undersampled with seed 42, leaving 7,788 posts per class (15,576 in total).
- Stratified partitions: 20% test; of the remainder, 80% forms the unlabeled pool and 20% is annotated; 20% of the annotated part is the validation set. This gives L = 1,993, V = 499, U = 9,968 and T = 3,116.

This is a balanced binary adaptation of HateXplain, not its official three-class benchmark. No normalized text is shared between partitions; near-duplicates are not checked.

In [1]:
# Install in the notebook's kernel. Keep the host-provided CUDA-compatible PyTorch.
%pip -q install "transformers==4.51.3" "huggingface-hub>=0.30,<1.0" "safetensors>=0.4" "scikit-learn>=1.5,<2" "scipy>=1.12,<2" "pandas>=2.2,<3" "matplotlib>=3.8,<4" "tqdm>=4.66" "nbformat>=5.10"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, sys, time, math, json, hashlib, random, gc, shutil, platform
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from IPython.display import display
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before running the experiments.'
DEVICE=torch.device('cuda')
HAS_GPU=True
print('GPU',torch.cuda.get_device_name(0))

GPU Tesla T4


In [3]:
ROOT = (Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd())/'hatexplain_benchmark_runs'
RESUME_FROM = None
HATEXPLAIN_LOCAL_SOURCE = None
RUN_LUDWIG = True  # existing BERT teacher/student implementation
if RESUME_FROM:
    source=Path(RESUME_FROM)
    assert source.is_dir() and source.resolve()!=ROOT.resolve()
    shutil.copytree(source,ROOT,dirs_exist_ok=True)
ROOT.mkdir(parents=True,exist_ok=True)

In [4]:
"""HateXplain loader: pinned source files, binary label mapping and balanced custom partitions."""
from collections import Counter
from pathlib import Path
import hashlib
import json
import re
import urllib.request
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

HATEXPLAIN_COMMIT = '01d742279dac941981f53806154481c0e15ee686'
HATEXPLAIN_SHA256 = {
    'dataset.json': '63bb3340fee0ec469b09690d04cb68f7c187787dd8b83807f071892c084967fb',
    'post_id_divisions.json': 'c2fb0d89862e7897b11ea3e9380753f15a793482b4b70ad0532dfb1212212835',
}

def hx_clean(text):
    t = re.sub(r'http\S+|www\.\S+', ' ', str(text))
    t = re.sub(r'@\w+', ' @user ', t)
    return re.sub(r'\s+', ' ', t).strip() or '[empty]'

def hx_digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()

def load_hatexplain(cache_dir, local_source=None, labeled_fraction=.20, seed=42):
    """Three-way majority, hate/offensive=1, normal=0; balanced custom splits.

    Partitions: 20% test; of the rest, 80% unlabeled and 20% annotated; 20% of the annotated part is validation.
    Remove normalized exact repeats/conflicts before sampling and splitting.
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    source_bytes = {}
    for filename, expected in HATEXPLAIN_SHA256.items():
        dest = cache_dir / filename
        if dest.exists():
            blob = dest.read_bytes()
        else:
            candidates = ([Path(local_source)/filename] if local_source else
                          sorted(Path('/kaggle/input').rglob(filename)))
            # Only accept a local file if its exact content matches the pinned official source.
            matches = [p for p in candidates if p.is_file() and
                       hashlib.sha256(p.read_bytes()).hexdigest() == expected]
            if matches:
                blob = matches[0].read_bytes()
            elif local_source:
                raise ValueError(f'{filename} missing or not the pinned official version in local_source')
            else:
                url = f'https://raw.githubusercontent.com/hate-alert/HateXplain/{HATEXPLAIN_COMMIT}/Data/{filename}'
                with urllib.request.urlopen(url, timeout=120) as response:
                    blob = response.read()
            if hashlib.sha256(blob).hexdigest() != expected:
                raise ValueError(f'Official source checksum mismatch: {filename}')
            tmp = dest.with_suffix('.tmp'); tmp.write_bytes(blob); tmp.replace(dest)
        if hashlib.sha256(blob).hexdigest() != expected:
            raise ValueError(f'Cached source checksum mismatch: {filename}')
        source_bytes[filename] = blob
    data = json.loads(source_bytes['dataset.json'])
    divisions = json.loads(source_bytes['post_id_divisions.json'])
    assert set(divisions) == {'train', 'val', 'test'}
    assigned = {}
    for split, ids in divisions.items():
        for post_id in ids:
            if post_id in assigned: raise ValueError('Official post ID appears in multiple splits')
            assigned[post_id] = split
    rows, ties = [], []
    for post_id, post in data.items():
        votes = Counter(str(a['label']).strip().lower() for a in post['annotators'])
        if not set(votes) <= {'hatespeech','offensive','normal'}:
            raise ValueError(f'Unknown annotation label for {post_id}')
        label, count = votes.most_common(1)[0]
        if count < 2:
            ties.append(post_id)
            if post_id in assigned: raise ValueError('No-majority item unexpectedly included in official split')
            continue
        if post_id not in assigned: raise ValueError('Majority item not assigned an official split')
        text = ' '.join(post['post_tokens'])
        rows.append({'post_id':post_id, 'text':text, 'y':int(label in {'hatespeech','offensive'}),
                     'source_label':label, 'official_split':assigned[post_id],
                     'normalized_text':hx_clean(text).casefold()})
    frame = pd.DataFrame(rows)
    original_sizes = frame.groupby('official_split').size().to_dict()
    assert original_sizes == {'test':1924, 'train':15383, 'val':1922}
    removed = []
    def drop_mask(part, mask, reason):
        for row in part.loc[mask].itertuples():
            removed.append({'post_id':row.post_id,'official_split':row.official_split,'reason':reason})
        return part.loc[~mask].copy()
    conflicting = frame.groupby('normalized_text').y.nunique()
    frame = drop_mask(frame, frame.normalized_text.isin(conflicting[conflicting>1].index),
                      'conflicting binary labels')
    frame = drop_mask(frame, frame.duplicated('normalized_text'), 'consistent normalized repeat')
    clean_counts = {str(k):int(v) for k,v in frame.y.value_counts().items()}
    n = int(frame.y.value_counts().min())
    balanced = pd.concat([g.sample(n=n,random_state=seed) for _,g in frame.groupby('y')])
    balanced = balanced.sample(frac=1,random_state=seed).reset_index(drop=True)
    dev, test = train_test_split(balanced,test_size=.20,stratify=balanced.y,random_state=seed)
    annotated, unlabeled = train_test_split(dev,test_size=1-labeled_fraction,stratify=dev.y,random_state=seed)
    labeled, val = train_test_split(annotated,test_size=.20,stratify=annotated.y,random_state=seed)
    labeled=labeled.assign(partition='train'); val=val.assign(partition='val')
    unlabeled=unlabeled.assign(partition='unl'); test=test.assign(partition='test')
    df = pd.concat([labeled, val, unlabeled, test], ignore_index=True)
    df['row_id'] = np.arange(len(df), dtype=np.int64)
    parts = {k:df[df.partition==k].reset_index(drop=True) for k in ('train','val','unl','test')}
    for a in parts:
        for b in parts:
            if a != b:
                assert not set(parts[a].normalized_text)&set(parts[b].normalized_text), (a,b)
    meta = {
        'adapter_version':'hatexplain-binary-balanced-custom-v2',
        'source_repository':'https://github.com/hate-alert/HateXplain',
        'source_commit':HATEXPLAIN_COMMIT, 'source_sha256':HATEXPLAIN_SHA256,
        'class_names':['Normal','Hate or offensive'],
        'label_rule':'original three-way majority first; hatespeech/offensive=1, normal=0',
        'no_majority_excluded':len(ties),'original_official_sizes':original_sizes,
        'retained_rows':len(df),'annotated_fraction_of_development':labeled_fraction,
        'split_seed':seed,'data_hash':hx_digest(df[['text','y']].values.tolist()),
        'post_id_hash':hx_digest(df.post_id.tolist()),
        'splits':{k:v.row_id.tolist() for k,v in parts.items()},
        'post_ids':{k:v.post_id.tolist() for k,v in parts.items()},
        'counts':{k:len(v) for k,v in parts.items()},
        'class_counts':{k:{str(int(c)):int(n) for c,n in v.y.value_counts().items()} for k,v in parts.items()},
        'removed_duplicate_rows':removed, 'clean_class_counts_before_balancing':clean_counts,
        'undersampled_rows':len(frame)-len(balanced),
        'official_test_unchanged':False,
        'normalized_duplicates_after_cleaning':int(df.duplicated('normalized_text').sum()),
        'policy':'custom balanced split: deduplicate, undersample majority, then historical stratified splits',
        'annotation_budget_note':'L+V is 20% of development; V is 20% of L+V; test is 20% of total',
    }
    return df, parts, meta

In [5]:
#@title 2. Configuration
# ---------------------------------------------------------------- what to run
DATASETS_TO_RUN = ['hatexplain']   # drop one to run a single dataset
RUN_FROZEN      = True     # sections 6-8: everything on the frozen SimCSE encoder
RUN_FINETUNE    = False     # section 9: Ludwig + proposed-policy-on-fine-tuned  (~2 h)
RUN_ABLATION    = False     # leave-one-component-out, per dataset  (~8 min each)
RUN_CV          = False     # 5-fold confirmation, per dataset      (~5 min each)
OPEN_TEST       = True     # the single pre-registered test evaluation per dataset

# ---------------------------------------------------------------- Kaggle paths
ORIGINAL_CSV = None        # not used in this notebook
CACHE_DIR    = str(ROOT/'cache')
OUT_DIR      = str(ROOT/'results')
CKPT_DIR     = str(ROOT/'checkpoints')
for d in (CACHE_DIR, OUT_DIR, CKPT_DIR): os.makedirs(d, exist_ok=True)

TEXT_COL, LABEL_COL = 'Content', 'Label'
CLASS_NAMES = ['Normal', 'Hate or offensive']

# ---------------------------------------------------------------- splits
RANDOM_STATE            = 42
TEST_SIZE               = 0.20
UNLABELED_FRACTION      = 0.80
VAL_FRACTION_OF_LABELED = 0.20

# ---------------------------------------------------------------- frozen encoder
SIMCSE_MODEL = 'princeton-nlp/sup-simcse-bert-base-uncased'
MAX_LENGTH, ENCODE_BATCH = 128, 64

# ---------------------------------------------------------------- the head
N_SEEDS      = 5
BEST_HIDDEN  = (768, 768, 256)
BEST_DROPOUT = 0.5
DECISION_THRESHOLD = 0.5

# ---------------------------------------------------------------- proposed method
TAU            = 0.95     # tier-1 base confidence, per-class adapted from here
TAU_FLOOR      = 0.80
EPI_Q          = 0.75     # reject the top quartile of epistemic uncertainty
TIER2_TAU      = 0.70
TIER2_LP_Q     = 0.50
TIER2_NBR      = 0.75
MIN_KNOWN_NBRS = 2
KNN_K          = 10
LP_ALPHA       = 0.99
SHARPEN_T      = 0.5
ROUNDS         = 3
PSEUDO_W_MAX   = 1.0
CLASS_BALANCE  = True

CV_FOLDS, CV_SEEDS, CV_ROUNDS = 5, 3, 2

# ---------------------------------------------------------------- Ludwig et al. (2024)
# Teacher-student baseline following Ludwig et al.: threshold 0.80, hard pseudo-labels, unit weights.
# Ludwig et al. use XLM-RoBERTa; this implementation uses bert-base-uncased.
LUDWIG = dict(model_name='bert-base-uncased', threshold=0.80,
              teacher_epochs=5, student_epochs=5, batch_size=32, lr=2e-5)
# the proposed conservative policy on the SAME backbone -- isolates policy from fine-tuning
FT_PROPOSED_CAP = 6000    # per class, most-confident-first
FT_PSEUDO_W     = 0.5

# ---------------------------------------------------------------- fingerprints
# Partition fingerprints of another dataset; not used for HateXplain.
EXPECTED_FP = {'train': '209632ba016850b8', 'val': '1468aae3253764ff',
               'unl': 'c16b71f16b85b3d6', 'test': '3b31d40b164f5675'}

TEST_LEDGER = []
print('configured | datasets:', DATASETS_TO_RUN)
print('           | RUN_FROZEN=%s RUN_FINETUNE=%s OPEN_TEST=%s'
      % (RUN_FROZEN, RUN_FINETUNE, OPEN_TEST))

configured | datasets: ['hatexplain']
           | RUN_FROZEN=True RUN_FINETUNE=False OPEN_TEST=True


In [6]:
hx_df,hx_parts,hx_meta=load_hatexplain(ROOT/'source_data',HATEXPLAIN_LOCAL_SOURCE,.20,42)
RAW={'hatexplain':hx_df.rename(columns={'text':TEXT_COL,'y':LABEL_COL})}
SPL={'hatexplain':{k:v.rename(columns={'text':TEXT_COL,'y':LABEL_COL}) for k,v in hx_parts.items()}}
YS={'hatexplain':{k:v.y.to_numpy(dtype=int) for k,v in hx_parts.items()}}
assert hx_meta['counts']=={'train':1993,'val':499,'unl':9968,'test':3116}
def stable_digest(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False,default=str).encode()).hexdigest()
def atomic_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');tmp.write_text(json.dumps(obj,indent=2,default=str));os.replace(tmp,path)
# Pin SimCSE to a local immutable HF snapshot. The original pooling function is retained.
SIMCSE_SNAPSHOT=snapshot_download(SIMCSE_MODEL,allow_patterns=['config.json','model.safetensors','pytorch_model.bin',
    'tokenizer.json','tokenizer_config.json','vocab.txt','special_tokens_map.json'])
BENCH_PROTOCOL={'revision':'hatexplain-contextual-v1','data':hx_meta,'simcse_model':SIMCSE_MODEL,
    'simcse_commit':Path(SIMCSE_SNAPSHOT).name,'full_rounds':3,'confidence_rounds':1,
    'confidence_tau':.95,'confidence_weight':.5,'final_labeled_pool':'L plus V',
    'ensemble_members':5,'source':'simcse-rf-fixed (2).ipynb','base_seed':42}
BENCH_ID=stable_digest(BENCH_PROTOCOL)[:16]
RUN_ROOT=ROOT/BENCH_ID;RUN_ROOT.mkdir(exist_ok=True)
CACHE_DIR=str(RUN_ROOT/'cache');OUT_DIR=str(RUN_ROOT/'results');CKPT_DIR=str(RUN_ROOT/'checkpoints')
for directory in (CACHE_DIR,OUT_DIR,CKPT_DIR):Path(directory).mkdir(parents=True,exist_ok=True)
atomic_json(RUN_ROOT/'protocol.json',BENCH_PROTOCOL)
atomic_json(ROOT/'hatexplain_manifest.json',hx_meta)
hx_df[['post_id','row_id','partition','text','y']].to_csv(ROOT/'hatexplain_partitions.csv',index=False)
print('Partition counts',hx_meta['counts'],'| benchmark protocol',BENCH_ID)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Partition counts {'train': 1993, 'val': 499, 'unl': 9968, 'test': 3116} | benchmark protocol 5b9c550ab5302f6f


In [7]:
#@title 5. Light preprocessing and frozen SimCSE features (cached)
def clean_light(text):
    """Strip URLs and @mentions, keep everything else.

    Casing, punctuation and word endings are preserved: a subword transformer was
    pretrained on natural text, and stemming or stopword removal here fights the
    tokenizer.
    """
    t = re.sub(r'http\S+|www\.\S+', ' ', str(text))
    t = re.sub(r'@\w+', ' @user ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t if t else '[empty]'

def _mean_pool(last_hidden, mask):
    m = mask.unsqueeze(-1).expand(last_hidden.size()).float()
    return torch.sum(last_hidden * m, 1) / torch.clamp(m.sum(1), min=1e-9)

@torch.no_grad()
def encode(texts, model_name=SIMCSE_MODEL, desc=''):
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
    out = []
    for i in tqdm(range(0, len(texts), ENCODE_BATCH), desc=desc or 'encoding', leave=False):
        e = tok(texts[i:i + ENCODE_BATCH], padding=True, truncation=True,
                max_length=MAX_LENGTH, return_tensors='pt')
        e = {k: v.to(DEVICE) for k, v in e.items()}
        h = mdl(**e, return_dict=True).last_hidden_state
        out.append(_mean_pool(h, e['attention_mask']).cpu().numpy())
    del mdl
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return np.vstack(out).astype(np.float32)

TXTS, FEATS = {}, {}
for ds in DATASETS_TO_RUN:
    TXTS[ds] = {k: [clean_light(t) for t in SPL[ds][k][TEXT_COL]] for k in SPL[ds]}
    FEATS[ds] = {}
    for k in ('train', 'val', 'unl', 'test'):
        p = '%s/simcse_mean_light__%s__%s.npy' % (CACHE_DIR, ds, k)
        if os.path.exists(p):
            FEATS[ds][k] = np.load(p); tag = 'cache'
        else:
            t0 = time.time()
            FEATS[ds][k] = encode(TXTS[ds][k], desc='%s/%s' % (ds, k))
            np.save(p, FEATS[ds][k]); tag = '%.0f s' % (time.time() - t0)
        print('  %-9s %-6s %-14s %s' % (ds, k, str(FEATS[ds][k].shape), tag))
print('\nfeatures ready')

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

hatexplain/train:   0%|          | 0/32 [00:00<?, ?it/s]

  hatexplain train  (1993, 768)    46 s


hatexplain/val:   0%|          | 0/8 [00:00<?, ?it/s]

  hatexplain val    (499, 768)     2 s


hatexplain/unl:   0%|          | 0/156 [00:00<?, ?it/s]

  hatexplain unl    (9968, 768)    36 s


hatexplain/test:   0%|          | 0/49 [00:00<?, ?it/s]

  hatexplain test   (3116, 768)    12 s

features ready


In [8]:
#@title 5. The head and its training loop
#
# The training loop supports two options used by the quality-aware selection. With hard
# labels and no clean_mask it is a standard supervised fit.
#
#   1. SOFT TARGETS. `y` may be (N,) integer labels or (N, C) probabilities. Soft
#      targets carry the teacher's uncertainty into the student's loss; hard labels
#      throw it away and let a 0.72-confidence pseudo-label assert itself as truth.
#
#   2. CLEAN-ONLY EARLY STOPPING. `clean_mask` marks which rows are ground truth. The
#      inner 10% early-stopping split is drawn ONLY from those, so the stopping
#      criterion is never measured against the pseudo-labels the model itself produced
#      -- otherwise a self-confirming model is rewarded for agreeing with itself.
#
# It also returns the inner-holdout predictions, which cost nothing and give a free
# out-of-fold set for temperature scaling in section 3.
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split as tts
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve,
                             precision_recall_fscore_support)
from scipy.stats import binomtest
import matplotlib.pyplot as plt

N_CLASSES = 2

class Head(nn.Module):
    """Linear -> BatchNorm1d -> GELU -> Dropout per block, then a linear classifier."""
    def __init__(self, d_in, hidden=(768, 768, 256), dropout=0.5, n_classes=N_CLASSES):
        super().__init__()
        layers, prev = [], d_in
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)


def train_head(Xtr, ytr, hidden=BEST_HIDDEN, dropout=BEST_DROPOUT, epochs=80, bs=256,
               lr=1e-3, wd=1e-2, seed=0, sample_weight=None, patience=12,
               clean_mask=None, return_oof=False, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    ytr = np.asarray(ytr)
    soft = ytr.ndim == 2
    y_hard = ytr.argmax(1) if soft else ytr.astype(int)
    n = len(y_hard)

    # inner early-stopping split, drawn only from ground-truth rows
    idx = np.arange(n)
    pool = idx if clean_mask is None else idx[np.asarray(clean_mask, dtype=bool)]
    assert len(pool) >= 20, 'not enough clean rows for an inner validation split'
    p_tr, p_va = tts(pool, test_size=0.10, stratify=y_hard[pool], random_state=seed)
    itr = np.setdiff1d(idx, p_va, assume_unique=False)   # everything except the holdout
    iva = p_va

    sc_ = StandardScaler().fit(Xtr[itr])
    Xa = torch.tensor(sc_.transform(Xtr[itr]), dtype=torch.float32, device=DEVICE)
    Xb = torch.tensor(sc_.transform(Xtr[iva]), dtype=torch.float32, device=DEVICE)
    yb = torch.tensor(y_hard[iva], dtype=torch.long, device=DEVICE)
    ya = (torch.tensor(ytr[itr], dtype=torch.float32, device=DEVICE) if soft
          else torch.tensor(y_hard[itr], dtype=torch.long, device=DEVICE))
    wa = (torch.tensor(np.asarray(sample_weight, dtype=np.float32)[itr], device=DEVICE)
          if sample_weight is not None else torch.ones(len(itr), device=DEVICE))

    mdl = Head(Xtr.shape[1], hidden, dropout).to(DEVICE)
    opt = torch.optim.AdamW(mdl.parameters(), lr=lr, weight_decay=wd)
    steps = max(1, int(np.ceil(len(itr) / bs))) * epochs
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.15)
    lossf = nn.CrossEntropyLoss(reduction='none')   # accepts soft targets in torch >= 1.10

    best, best_state, bad, ep = -1.0, None, 0, 0
    for ep in range(epochs):
        mdl.train()
        perm = torch.randperm(len(itr), device=DEVICE)
        for i in range(0, len(itr), bs):
            j = perm[i:i + bs]
            if len(j) < 2: continue          # BatchNorm needs more than one sample
            opt.zero_grad(set_to_none=True)
            loss = (lossf(mdl(Xa[j]), ya[j]) * wa[j]).mean()
            loss.backward(); opt.step(); sch.step()
        mdl.eval()
        with torch.no_grad():
            acc = (mdl(Xb).argmax(1) == yb).float().mean().item()
        if acc > best:
            best, bad = acc, 0
            best_state = {k: v.detach().clone() for k, v in mdl.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    if best_state is not None: mdl.load_state_dict(best_state)
    mdl.eval()
    if verbose: print('    inner-val %.4f, stopped at epoch %d' % (best, ep + 1))
    if return_oof:
        with torch.no_grad():
            oof = torch.softmax(mdl(Xb), -1).cpu().numpy()
        return mdl, sc_, (iva, oof, y_hard[iva])
    return mdl, sc_


@torch.no_grad()
def head_proba(mdl, sc_, X, bs=8192):
    out = []
    for i in range(0, len(X), bs):
        t = torch.tensor(sc_.transform(X[i:i + bs]), dtype=torch.float32, device=DEVICE)
        out.append(torch.softmax(mdl(t), -1).cpu().numpy())
    return np.vstack(out)


def members_proba(models, X):
    """(M, N, C) -- the per-seed probabilities the uncertainty decomposition needs."""
    return np.stack([head_proba(m, s, X) for m, s in models], 0)


def ens_proba(models, X): return members_proba(models, X).mean(0)


def fit_ensemble(X, y, w=None, seeds=N_SEEDS, clean_mask=None, return_oof=False, **kw):
    models, oofs = [], []
    for sd in range(seeds):
        r = train_head(X, y, seed=sd, sample_weight=w, clean_mask=clean_mask,
                       return_oof=return_oof, **kw)
        if return_oof:
            m, s, o = r; oofs.append(o)
        else:
            m, s = r
        models.append((m, s))
    return (models, oofs) if return_oof else models


def mcnemar(y, pred_a, pred_b, name_a='A', name_b='B', quiet=False):
    y = np.asarray(y); a_ok, b_ok = (pred_a == y), (pred_b == y)
    b_ = int((a_ok & ~b_ok).sum()); c_ = int((~a_ok & b_ok).sum())
    p = binomtest(min(b_, c_), b_ + c_, 0.5).pvalue if (b_ + c_) else 1.0
    if not quiet:
        print('  %s only correct: %d | %s only correct: %d | discordant %d'
              % (name_a, b_, name_b, c_, b_ + c_))
        print('  net %+d in favour of %s | McNemar exact p = %.3g -> %s'
              % (b_ - c_, name_a, p, 'SIGNIFICANT' if p < 0.05 else 'not significant'))
    return {'b': b_, 'c': c_, 'p': float(p)}


def bootstrap_ci(y, yp, metric='macro_f1', n_boot=2000, seed=RANDOM_STATE):
    y, yp = np.asarray(y), np.asarray(yp)
    rng = np.random.RandomState(seed); n = len(y); v = np.empty(n_boot)
    for b in range(n_boot):
        i = rng.randint(0, n, n)
        v[b] = (f1_score(y[i], yp[i], average='macro') if metric == 'macro_f1'
                else accuracy_score(y[i], yp[i]))
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))


def score(y, yp, name='', quiet=False):
    a = accuracy_score(y, yp); f = f1_score(y, yp, average='macro')
    if not quiet: print('%-38s acc %.4f  macroF1 %.4f' % (name, a, f))
    return a, f


n_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
TEST_OPENED = False
print('defined | device', DEVICE)

defined | device cuda


In [9]:
#@title 2. Calibration, uncertainty decomposition, alignment, thresholds
from scipy.optimize import minimize_scalar
from scipy.sparse import csr_matrix, identity, diags
from scipy.sparse.linalg import cg

EPS = 1e-12

# ----------------------------------------------------------------- calibration
def fit_temperature(P, y, lo=0.25, hi=8.0):
    """One-parameter temperature on the ensemble mean probability.

    Every threshold in this method is a statement about probability, so probability
    has to mean something first. Fits T by minimising NLL of softmax(log P / T).
    T = 1 lies inside the search interval, so the fit can never increase NLL.
    """
    P = np.clip(np.asarray(P, dtype=np.float64), EPS, 1.0)
    logp = np.log(P); y = np.asarray(y); n = len(y)
    def nll(T):
        z = logp / T; z = z - z.max(1, keepdims=True)
        e = np.exp(z); q = e / e.sum(1, keepdims=True)
        return -np.log(q[np.arange(n), y] + EPS).mean()
    return float(minimize_scalar(nll, bounds=(lo, hi), method='bounded').x)

def apply_temperature(P, T):
    P = np.clip(np.asarray(P, dtype=np.float64), EPS, 1.0)
    z = np.log(P) / T; z = z - z.max(1, keepdims=True)
    e = np.exp(z); return e / e.sum(1, keepdims=True)

def ece(P, y, n_bins=15):
    P = np.asarray(P); y = np.asarray(y)
    conf, pred = P.max(1), P.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1); out = 0.0
    for i in range(n_bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1])
        if m.sum(): out += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(out)

def nll_of(P, y):
    P = np.clip(np.asarray(P), EPS, 1.0); y = np.asarray(y)
    return float(-np.log(P[np.arange(len(y)), y]).mean())

# ------------------------------------------------------ uncertainty decomposition
def _H(p, axis=-1):
    p = np.clip(p, EPS, 1.0); return -(p * np.log(p)).sum(axis)

def decompose_uncertainty(P_members):
    """P_members: (M, N, C).

        total     = H[E_m p_m]          predictive entropy
        aleatoric = E_m H[p_m]          irreducible: the sample is genuinely ambiguous
        epistemic = total - aleatoric   BALD mutual information: the seeds disagree

    Max-softmax conflates the two. A sample can be confident on average while the seeds
    violently disagree about it, and those are exactly the confidently-wrong pseudo-labels
    that a plain confidence threshold cannot filter out.
    """
    P_members = np.asarray(P_members, dtype=np.float64)
    Pbar = P_members.mean(0)
    total = _H(Pbar); aleatoric = _H(P_members).mean(0)
    return Pbar, total, aleatoric, np.maximum(total - aleatoric, 0.0)

# ------------------------------------------ distribution alignment and thresholds
def distribution_align(P, prior, running=None, momentum=0.0):
    """ReMixMatch alignment: pull the pseudo-label marginal back onto the known prior.

    This is the direct fix for the drift behind the measured purity decay
    (0.9641 -> 0.9532 -> 0.9353): once the accepted set skews to one class, the next
    teacher is trained on a skewed set and skews further.
    """
    P = np.asarray(P, dtype=np.float64); cur = P.mean(0)
    running = cur if running is None else momentum * running + (1 - momentum) * cur
    Q = P * (np.asarray(prior, dtype=np.float64) / np.clip(running, EPS, None))
    return Q / np.clip(Q.sum(1, keepdims=True), EPS, None), running

def flexmatch_thresholds(P, tau, n_classes=N_CLASSES, tau_floor=0.80, convex=False):
    """Per-class curriculum thresholds, in the spirit of FlexMatch.

    sigma_c counts how many unlabelled samples clear the base threshold as class c; the
    class the model has learned less about gets a lower bar, so the accepted set does not
    silently skew towards the easy class.

    Two departures from the paper, because the teacher here is an already-converged
    ensemble rather than a network early in training: we normalise by sigma.max() alone
    (the paper's extra term exists to loosen every threshold during warm-up, which here
    just collapses them), and we clip at `tau_floor`.
    """
    P = np.asarray(P); conf, pred = P.max(1), P.argmax(1)
    passed = conf >= tau
    sigma = np.array([int(((pred == c) & passed).sum()) for c in range(n_classes)], float)
    beta = sigma / max(sigma.max(), 1.0)
    warp = beta / (2.0 - beta) if convex else beta
    return np.clip(tau * np.clip(warp, 1e-3, 1.0), tau_floor, tau), sigma

def sharpen(P, T=0.5):
    P = np.clip(np.asarray(P, dtype=np.float64), EPS, 1.0) ** (1.0 / T)
    return P / P.sum(1, keepdims=True)

def ramp(r, n_rounds):
    """Unlabelled-weight ramp. The first teacher is the least trustworthy one."""
    if n_rounds <= 1: return 1.0
    t = min(max(r / float(n_rounds), 0.0), 1.0)
    return float(np.exp(-5.0 * (1.0 - t) ** 2))

def coverage_matched_purity(conf, pred, y_true, target_coverage):
    """Purity a PLAIN threshold reaches at the SAME coverage.

    The only fair comparison for a selection rule. "Purity 0.96" is not a result on its
    own -- the plain rule also reaches 0.96, at some coverage. The claim has content only
    if, at matched coverage, the new rule is purer.
    """
    conf = np.asarray(conf)
    thr = float(np.quantile(conf, 1.0 - float(np.clip(target_coverage, 0, 1))))
    m = conf >= thr
    if not m.any(): return thr, float('nan'), 0.0
    return thr, float((np.asarray(pred)[m] == np.asarray(y_true)[m]).mean()), float(m.mean())

print('selection machinery defined')

selection machinery defined


In [10]:
#@title 3. k-NN graph and label propagation
def knn_graph(X, k=KNN_K, gamma=3.0, chunk=1024, use_torch=None):
    """Cosine k-NN graph with relu(cos)^gamma affinities (Iscen et al., 2019).

    X is L2-normalised, so a chunked matmul gives cosine similarity directly. On a GPU
    this is ~10 s for 53k x 768; on CPU allow a few minutes.
    """
    X = np.asarray(X, dtype=np.float32)
    Xn = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), EPS, None)
    n = Xn.shape[0]
    use_torch = HAS_GPU if use_torch is None else use_torch
    rows = np.repeat(np.arange(n), k)
    cols = np.empty(n * k, dtype=np.int64); vals = np.empty(n * k, dtype=np.float32)
    if use_torch:
        Xt = torch.tensor(Xn, device=DEVICE)
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        if use_torch:
            sim = (Xt[s:e] @ Xt.T)
            sim[torch.arange(e - s, device=DEVICE), torch.arange(s, e, device=DEVICE)] = -2.0
            v, i = torch.topk(sim, k, dim=1)
            idx = i.cpu().numpy(); srt = v.cpu().numpy()
        else:
            sim = Xn[s:e] @ Xn.T
            sim[np.arange(e - s), np.arange(s, e)] = -2.0
            idx = np.argpartition(-sim, k, axis=1)[:, :k]
            srt = np.take_along_axis(sim, idx, 1)
            o = np.argsort(-srt, axis=1)
            idx = np.take_along_axis(idx, o, 1); srt = np.take_along_axis(srt, o, 1)
        cols[s * k:e * k] = idx.ravel()
        vals[s * k:e * k] = np.maximum(srt, 0.0).ravel() ** gamma
    W = csr_matrix((vals, (rows, cols)), shape=(n, n))
    W = W.maximum(W.T)                      # symmetrise
    W.setdiag(0.0); W.eliminate_zeros()
    return W


def label_propagation(W, y_lab, lab_idx, n_all, n_classes=N_CLASSES, alpha=LP_ALPHA,
                      tol=1e-6, maxiter=200):
    """Zhou et al. label spreading: F = (I - alpha*S)^-1 Y, S = D^-1/2 W D^-1/2.

    Solved per class with conjugate gradient. Returns row-normalised class scores and an
    entropy-derived certainty in [0, 1].
    """
    d = np.asarray(W.sum(1)).ravel()
    Dm = diags(1.0 / np.sqrt(np.clip(d, EPS, None)))
    A = (identity(n_all, format='csr') - alpha * (Dm @ W @ Dm)).tocsr()
    Ymat = np.zeros((n_all, n_classes))
    Ymat[np.asarray(lab_idx), np.asarray(y_lab)] = 1.0
    Fp = np.zeros((n_all, n_classes))
    for c in range(n_classes):
        try:
            f, _ = cg(A, Ymat[:, c], rtol=tol, maxiter=maxiter)
        except TypeError:                    # scipy < 1.12 spells it `tol`
            f, _ = cg(A, Ymat[:, c], tol=tol, maxiter=maxiter)
        Fp[:, c] = f
    Fp = np.clip(Fp, 0.0, None)
    Fn = Fp / np.clip(Fp.sum(1, keepdims=True), EPS, None)
    cert = np.clip(1.0 - _H(Fn) / np.log(n_classes), 0.0, 1.0)
    return Fn, cert


def neighbourhood_purity(W, labels, mask_known, n_classes=N_CLASSES):
    """Fraction of a node's KNOWN-labelled neighbours that agree with its own label.

    A cheap, high-precision noise filter: a pseudo-label sitting in a class-mixed
    neighbourhood is far likelier to be wrong than its softmax admits. Returns
    (purity, n_known_neighbours) -- the count matters because purity over one neighbour
    is not evidence.
    """
    Wb = (W > 0).astype(np.float32).tocsr()
    n = W.shape[0]
    onehot = np.zeros((n, n_classes), dtype=np.float32)
    known = np.asarray(mask_known, dtype=bool)
    onehot[known, np.asarray(labels)[known]] = 1.0
    nb = Wb @ onehot
    tot = nb.sum(1)
    own = nb[np.arange(n), np.asarray(labels)]
    return own / np.clip(tot, EPS, None), tot


# ---- build the graph once over train + val + unl (test is never a node, so the final
# ---- classifier stays inductive: nothing about the test set enters training)
print('graph functions defined')

graph functions defined


In [11]:
#@title 4. Two-tier acceptance and the iterative runner
def select_pseudo_labels(Pcal, epistemic, lp_scores=None, lp_cert=None, nbr_pur=None,
                         nbr_cnt=None, tau=TAU, tau_floor=TAU_FLOOR, epi_q=EPI_Q,
                         tier2_tau=TIER2_TAU, tier2_lp_q=TIER2_LP_Q, tier2_nbr=TIER2_NBR,
                         min_known_nbrs=MIN_KNOWN_NBRS, class_balance=True,
                         use_flexmatch=True, n_classes=N_CLASSES):
    """Returns (accept_mask, hard_pred, tier) with tier in {0, 1, 2}."""
    Pcal = np.asarray(Pcal); n = len(Pcal)
    conf, pred = Pcal.max(1), Pcal.argmax(1)

    tau_c = (flexmatch_thresholds(Pcal, tau, n_classes, tau_floor)[0] if use_flexmatch
             else np.full(n_classes, tau))
    tier1 = conf >= tau_c[pred]
    if epi_q < 1.0:
        tier1 &= epistemic <= np.quantile(epistemic, epi_q)

    tier2 = np.zeros(n, dtype=bool)
    if lp_scores is not None:
        ok = (np.asarray(lp_scores).argmax(1) == pred) & (conf >= tier2_tau)
        if lp_cert is not None:
            # rank-based, not absolute: the propagated-score scale depends on graph
            # density and alpha, so a fixed cut-off would not be portable
            c = np.asarray(lp_cert); ok &= c >= np.quantile(c, tier2_lp_q)
        if nbr_pur is not None: ok &= np.asarray(nbr_pur) >= tier2_nbr
        if nbr_cnt is not None: ok &= np.asarray(nbr_cnt) >= min_known_nbrs
        tier2 = ok & ~tier1

    accept = tier1 | tier2
    tier = np.where(tier1, 1, np.where(tier2, 2, 0))

    if class_balance and accept.any():
        cnt = np.array([int((accept & (pred == c)).sum()) for c in range(n_classes)])
        if cnt.min() > 0 and cnt.max() > cnt.min():
            keep = np.zeros(n, dtype=bool)
            for c in range(n_classes):
                ic = np.where(accept & (pred == c))[0]
                keep[ic[np.argsort(-conf[ic])][:cnt.min()]] = True
            accept, tier = keep, np.where(keep, tier, 0)
    return accept, pred, tier


def pseudo_weights(Pcal, accept, tier, lam, tier2_scale=0.75, floor=0.10,
                   n_classes=N_CLASSES):
    """w = lam * normalised calibrated margin, tier-2 discounted. Not a flat 0.5.

    A flat weight gives a p=0.999 pseudo-label and a p=0.72 one the same weight. Here the
    loss degrades gracefully with the teacher's own stated confidence.
    """
    conf = np.asarray(Pcal).max(1)
    m = np.clip((conf - 1.0 / n_classes) / (1.0 - 1.0 / n_classes), 0.0, 1.0)
    w = lam * np.clip(m, floor, 1.0)
    return np.where(accept, np.where(tier == 2, w * tier2_scale, w), 0.0)


def run_ssl(X_lab, y_lab, lab_nodes, unl_nodes, rounds=ROUNDS, seeds=N_SEEDS,
            calibrate=True, use_epistemic=True, use_graph=True, soft_targets=True,
            graded_weights=True, class_balance=CLASS_BALANCE, align=True, relabel=True,
            tau=TAU, epi_q=EPI_Q, flat_weight=0.5, X_val=None, y_val=None,
            y_unl_true=None, tag='', quiet=False, base=None):
    """The method. Every component is a flag, so section 7 can ablate each one.

    calibrate=False, use_epistemic=False, use_graph=False, soft_targets=False,
    graded_weights=False, class_balance=False, align=False, relabel=False, rounds=1
    reproduces the plain confidence-threshold rule (the "SimCSE confidence" system).
    """
    F_unl = F['unl']; n_unl = len(F_unl)
    y_lab = np.asarray(y_lab)
    prior = np.bincount(y_lab, minlength=N_CLASSES) / len(y_lab)

    # ---- graph evidence: depends only on the labelled seed set, so computed once ----
    lp_s = lp_c = None
    if use_graph:
        lp_full, cert_full = label_propagation(W, y_lab, lab_nodes, N_NODES)
        lp_s, lp_c = lp_full[unl_nodes], cert_full[unl_nodes]

    # ---- supervised teacher, plus a free out-of-fold set for calibration ----
    # The inner early-stopping holdouts are already computed by train_head, they are
    # ground truth (clean_mask keeps pseudo-labels out of them), and using them means
    # clean_val is never spent on calibration -- so val stays an honest selection set.
    cur, oofs = base if base is not None else fit_ensemble(X_lab, y_lab, seeds=seeds,
                                                          return_oof=True)
    def temp(o):
        return fit_temperature(np.vstack([x[1] for x in o]),
                               np.concatenate([x[2] for x in o])) if calibrate else 1.0
    T = temp(oofs)
    if not quiet and calibrate:
        print('  %scalibration T = %.3f (from %s out-of-fold predictions)'
              % (tag, T, format(sum(len(x[2]) for x in oofs), ',')))

    accepted = np.zeros(n_unl, bool)
    pseudo = np.zeros(n_unl, int)
    pseudo_soft = np.zeros((n_unl, N_CLASSES))
    pseudo_w = np.zeros(n_unl)
    node_lab = np.zeros(N_NODES, int); node_known = np.zeros(N_NODES, bool)
    node_lab[lab_nodes] = y_lab; node_known[lab_nodes] = True

    running, hist = None, []
    if X_val is not None:
        a0, f0 = score(y_val, ens_proba(cur, X_val).argmax(1), quiet=True)
        hist.append({'round': 0, 'new': 0, 'total': 0, 'coverage': 0.0, 'tier1': 0,
                     'tier2': 0, 'purity': np.nan, 'lambda': 0.0,
                     'val acc': a0, 'val macroF1': f0})
        if not quiet:
            print('  %sround 0 (supervised): val acc %.4f  macroF1 %.4f' % (tag, a0, f0))

    for r in range(1, rounds + 1):
        Pbar, _, _, epi = decompose_uncertainty(members_proba(cur, F_unl))
        Pcal = apply_temperature(Pbar, T) if calibrate else Pbar
        if align:
            Pcal, running = distribution_align(Pcal, prior, running, momentum=0.0)

        nb_p = nb_c = None
        if use_graph:
            np_, nc_ = neighbourhood_purity(W, node_lab, node_known)
            nb_p, nb_c = np_[unl_nodes], nc_[unl_nodes]

        sel, pred, tier = select_pseudo_labels(
            Pcal, epi, lp_scores=lp_s, lp_cert=lp_c, nbr_pur=nb_p, nbr_cnt=nb_c,
            tau=tau, epi_q=epi_q if use_epistemic else 1.0,
            class_balance=class_balance, use_flexmatch=align)

        lam = ramp(r, rounds) * PSEUDO_W_MAX if graded_weights else 1.0
        w_now = (pseudo_weights(Pcal, sel, tier, lam) if graded_weights
                 else np.where(sel, flat_weight, 0.0))
        soft_now = sharpen(Pcal, SHARPEN_T)

        new = sel & ~accepted
        if relabel:
            # nothing is frozen: every round re-decides the whole pool, so a label the
            # model has since changed its mind about is corrected instead of compounding
            accepted = sel.copy()
            pseudo[sel] = pred[sel]; pseudo_soft[sel] = soft_now[sel]; pseudo_w[sel] = w_now[sel]
        else:
            # confidence-only behaviour: once accepted, the label and weight are fixed
            accepted |= new
            pseudo[new] = pred[new]; pseudo_soft[new] = soft_now[new]; pseudo_w[new] = w_now[new]
            tier = np.where(accepted, np.maximum(tier, 1), 0)

        if accepted.sum() == 0:
            if not quiet: print('  %sround %d: nothing accepted -- stopping' % (tag, r))
            break

        Xa = np.vstack([X_lab, F_unl[accepted]])
        ya = (np.vstack([np.eye(N_CLASSES)[y_lab], pseudo_soft[accepted]]) if soft_targets
              else np.concatenate([y_lab, pseudo[accepted]]))
        wa = np.concatenate([np.ones(len(X_lab)), pseudo_w[accepted]])
        cm = np.concatenate([np.ones(len(X_lab), bool), np.zeros(int(accepted.sum()), bool)])
        cur, oofs = fit_ensemble(Xa, ya, w=wa, seeds=seeds, clean_mask=cm, return_oof=True)
        T = temp(oofs)                      # the teacher changed, so recalibrate it

        node_known[unl_nodes] = False
        node_lab[unl_nodes[accepted]] = pseudo[accepted]
        node_known[unl_nodes[accepted]] = True

        pur = (float((pseudo[accepted] == np.asarray(y_unl_true)[accepted]).mean())
               if y_unl_true is not None else np.nan)
        row = {'round': r, 'new': int(new.sum()), 'total': int(accepted.sum()),
               'coverage': float(accepted.mean()),
               'tier1': int((tier == 1).sum()), 'tier2': int((tier == 2).sum()),
               'purity': pur, 'lambda': float(lam)}
        if X_val is not None:
            a, f = score(y_val, ens_proba(cur, X_val).argmax(1), quiet=True)
            row['val acc'], row['val macroF1'] = a, f
            if not quiet:
                print('  %sround %d: %s accepted (%.1f%% cov, t1 %s / t2 %s, purity %.4f,'
                      ' lam %.2f) -> val %.4f / F1 %.4f'
                      % (tag, r, format(int(accepted.sum()), ','), accepted.mean() * 100,
                         format(int((tier == 1).sum()), ','),
                         format(int((tier == 2).sum()), ','), pur, lam, a, f))
        hist.append(row)

    return cur, pd.DataFrame(hist), accepted, pseudo, (lp_s, lp_c)


print('acceptance rule and runner defined')

acceptance rule and runner defined


In [12]:
#@title 7. Per-dataset activation, and checkpointing
# run_ssl and the graph functions read module-level globals (F, Y, W, N_NODES, IDX).
# activate() swaps those to one dataset so the identical code runs on both.
GRAPH_ORDER = ('train', 'val', 'unl')
_GRAPH_CACHE = {}
ACTIVE = None

def activate(ds, quiet=False):
    global F, Y, IDX, W, N_NODES, OFF, PRIOR, TXT, ACTIVE, X_GRAPH
    if ACTIVE == ds: return
    F, Y, TXT = FEATS[ds], YS[ds], TXTS[ds]
    OFF, _o = {}, 0
    for _k in GRAPH_ORDER:
        OFF[_k] = _o; _o += len(Y[_k])
    N_NODES = _o
    IDX = {k: np.arange(OFF[k], OFF[k] + len(Y[k])) for k in GRAPH_ORDER}
    PRIOR = np.bincount(Y['train'], minlength=N_CLASSES) / len(Y['train'])
    if ds not in _GRAPH_CACHE:
        t0 = time.time()
        X_GRAPH = np.vstack([F[k] for k in GRAPH_ORDER])
        _GRAPH_CACHE[ds] = knn_graph(X_GRAPH, k=KNN_K)
        if not quiet:
            print('[%s] graph: %s nodes, %s edges, k=%d, %.0f s'
                  % (ds, format(N_NODES, ','), format(_GRAPH_CACHE[ds].nnz, ','),
                     KNN_K, time.time() - t0))
        del X_GRAPH
    W = _GRAPH_CACHE[ds]
    ACTIVE = ds
    if not quiet:
        print('[%s] active | train %s val %s unl %s test %s | prior %s'
              % (ds, *[format(len(Y[k]), ',') for k in ('train','val','unl','test')],
                 np.round(PRIOR, 4)))

# ---------------------------------------------------------------- checkpoints
def ckpt_path(ds, stage): return '%s/%s__%s.json' % (CKPT_DIR, ds, stage)

def ckpt_save(ds, stage, obj):
    def enc(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray):     return o.tolist()
        if isinstance(o, pd.DataFrame):   return o.to_dict('records')
        return str(o)
    json.dump(obj, open(ckpt_path(ds, stage), 'w'), indent=1, default=enc)
    print('   [ckpt] saved %s/%s' % (ds, stage))

def ckpt_load(ds, stage):
    p = ckpt_path(ds, stage)
    return json.load(open(p)) if os.path.exists(p) else None

def save_preds(ds, name, y_pred, proba=None):
    tag = re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_')
    np.save('%s/preds__%s__%s.npy' % (OUT_DIR, ds, tag), np.asarray(y_pred))
    if proba is not None:
        np.save('%s/proba__%s__%s.npy' % (OUT_DIR, ds, tag), np.asarray(proba))

print('activation and checkpointing ready')

activation and checkpointing ready


## SimCSE systems

Both systems start from the same supervised ensemble fitted on L ∪ V. Reference labels of the unlabeled pool are not passed to the selection; only model predictions are used. Each completed system saves its predictions, ensemble weights and scalers.

In [13]:
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True

def save_ensemble(models,path):
    # Store standardization parameters numerically, without opaque pickled sklearn estimators.
    for i,(model,scaler) in enumerate(models):
        dest=path/f'member_{i}';dest.mkdir(parents=True,exist_ok=True)
        torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},dest/'weights.pt')
        np.savez_compressed(dest/'scaler.npz',mean=scaler.mean_,scale=scaler.scale_,var=scaler.var_)

def compact_simcse():
    ds='hatexplain';activate(ds)
    X_lab=np.vstack([F['train'],F['val']]); y_lab=np.r_[Y['train'],Y['val']]
    nodes=np.r_[IDX['train'],IDX['val']];unl_nodes=IDX['unl']
    reference=None; results=[]
    settings=[('SimCSE confidence',dict(rounds=1,calibrate=False,use_epistemic=False,use_graph=False,
                soft_targets=False,graded_weights=False,class_balance=False,align=False,relabel=False,
                tau=.95,flat_weight=.5)),
              ('SimCSE full',dict(rounds=3))]
    for name,kwargs in settings:
        tag=re.sub('[^a-z0-9]+','_',name.lower()).strip('_');dest=RUN_ROOT/tag;dest.mkdir(exist_ok=True)
        complete=dest/'complete.json'
        if complete.exists():
            results.append(json.loads(complete.read_text()));print('Reusing',name);continue
        if reference is None:
            seed_all(42)
            reference=fit_ensemble(X_lab,y_lab,return_oof=True)
        seed_all(42);started=time.time()
        models,history,mask,pseudo,_=run_ssl(X_lab,y_lab,nodes,unl_nodes,base=reference,
                                          y_unl_true=None,tag=name+' ',**kwargs)
        P=ens_proba(models,F['test']);pred=P.argmax(1);yt=np.asarray(Y['test'])
        test_frame=hx_parts['test']
        np.savez_compressed(dest/'predictions.npz',row_id=test_frame.row_id.to_numpy(),
            post_id=test_frame.post_id.to_numpy(dtype=str),y_true=yt,probabilities=P,y_pred=pred)
        pd.DataFrame({'post_id':test_frame.post_id,'row_id':test_frame.row_id,'true':yt,'pred':pred,
                      'p_hate_or_offensive':P[:,1]}).to_csv(dest/'predictions.csv',index=False)
        save_ensemble(models,dest/'ensemble')
        history.to_csv(dest/'round_history.csv',index=False)
        lo,hi=bootstrap_ci(yt,pred,'macro_f1')
        result={'dataset':'HateXplain','method':name,'macro_f1':float(f1_score(yt,pred,average='macro',zero_division=0)),
                'accuracy':float(accuracy_score(yt,pred)),'roc_auc':float(roc_auc_score(yt,P[:,1])),
                'pseudo_labels':int(mask.sum()),'coverage':float(mask.mean()),'f1_ci_low':lo,'f1_ci_high':hi,
                'repetitions':1,'ensemble_members':5,'labeled_pool':'L+V','labeled_rows':len(y_lab),
                'requested_rounds':kwargs['rounds'],'completed_rounds':len(history),
                'seconds_excluding_shared_initial_ensemble':time.time()-started,'benchmark_id':BENCH_ID}
        pd.DataFrame(classification_report(yt,pred,labels=[0,1],target_names=CLASS_NAMES,
            output_dict=True,zero_division=0)).T.to_csv(dest/'classification_report.csv')
        pd.DataFrame(confusion_matrix(yt,pred,labels=[0,1])).to_csv(dest/'confusion_matrix.csv',index=False)
        atomic_json(complete,result);results.append(result);display(pd.DataFrame([result]))
        del models;gc.collect();torch.cuda.empty_cache()
    return results

SIMCSE_RESULTS=compact_simcse()
pd.DataFrame(SIMCSE_RESULTS).to_csv(RUN_ROOT/'table6_simcse_benchmarks.csv',index=False)
print('Saved',RUN_ROOT/'table6_simcse_benchmarks.csv')

[hatexplain] graph: 12,460 nodes, 211,274 edges, k=10, 0 s
[hatexplain] active | train 1,993 val 499 unl 9,968 test 3,116 | prior [0.4997 0.5003]


,dataset,method,macro_f1,accuracy,roc_auc,pseudo_labels,coverage,f1_ci_low,f1_ci_high,repetitions,ensemble_members,labeled_pool,labeled_rows,requested_rounds,completed_rounds,seconds_excluding_shared_initial_ensemble,benchmark_id
0,HateXplain,SimCSE confidence,0.728175,0.728177,0.804222,26,0.002608,0.712892,0.743219,1,5,L+V,2492,1,1,6.552698,5b9c550ab5302f6f


  SimCSE full calibration T = 2.555 (from 1,250 out-of-fold predictions)


,dataset,method,macro_f1,accuracy,roc_auc,pseudo_labels,coverage,f1_ci_low,f1_ci_high,repetitions,ensemble_members,labeled_pool,labeled_rows,requested_rounds,completed_rounds,seconds_excluding_shared_initial_ensemble,benchmark_id
0,HateXplain,SimCSE full,0.721327,0.721438,0.800835,996,0.09992,0.705656,0.736753,1,5,L+V,2492,3,3,13.163382,5b9c550ab5302f6f


Saved /kaggle/working/hatexplain_benchmark_runs/5b9c550ab5302f6f/table6_simcse_benchmarks.csv


## Ludwig-style teacher–student baseline

A `bert-base-uncased` teacher is fine-tuned on L for five epochs and labels the unlabeled pool. Predictions with confidence of at least 0.80 are kept as hard pseudo-labels with unit weight, and a student is trained for five epochs on L ∪ V and the pseudo-labeled examples.

The printed purity of the accepted pseudo-labels uses the reference labels of the unlabeled pool. It is a diagnostic only and does not affect training or selection.

In [14]:
# ---------------------- fine-tuning utilities (teacher/student) ----------------------
from torch.utils.data import Dataset, DataLoader

class TextDS(Dataset):
    def __init__(self, texts, labels, tok, max_len=MAX_LENGTH, weights=None):
        self.texts = list(texts); self.labels = np.asarray(labels)
        self.tok = tok; self.max_len = max_len
        self.weights = np.ones(len(self.labels)) if weights is None else np.asarray(weights)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        e = self.tok(self.texts[i], truncation=True, max_length=self.max_len,
                     padding='max_length', return_tensors='pt')
        return {'input_ids': e['input_ids'][0], 'attention_mask': e['attention_mask'][0],
                'labels': torch.tensor(int(self.labels[i])),
                'w': torch.tensor(float(self.weights[i]))}

def finetune(model_name, texts, labels, epochs, batch_size, lr, weights=None, tag=''):
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=int(len(np.unique(labels)))).to(DEVICE)
    dl = DataLoader(TextDS(texts, labels, tok, weights=weights),
                    batch_size=batch_size, shuffle=True, num_workers=2)
    opt = torch.optim.AdamW(mdl.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr,
                                                total_steps=epochs * len(dl), pct_start=0.1)
    lossf = torch.nn.CrossEntropyLoss(reduction='none')
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
    mdl.train()
    for ep in range(epochs):
        run, seen = 0.0, 0
        bar = tqdm(dl, desc=f'{tag} epoch {ep+1}/{epochs}')
        for b in bar:
            opt.zero_grad(set_to_none=True)
            ids, am = b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE)
            yb, wb = b['labels'].to(DEVICE), b['w'].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                loss = (lossf(mdl(input_ids=ids, attention_mask=am).logits, yb) * wb).mean()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item() * len(yb); seen += len(yb)
            bar.set_postfix(loss=f'{run/seen:.4f}')
    return mdl, tok

@torch.no_grad()
def predict_proba_ft(mdl, tok, texts, batch_size=128):
    mdl.eval(); out = []
    for i in tqdm(range(0, len(texts), batch_size), desc='inference', leave=False):
        e = tok(texts[i:i+batch_size], truncation=True, max_length=MAX_LENGTH,
                padding=True, return_tensors='pt')
        e = {k: v.to(DEVICE) for k, v in e.items()}
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
            logits = mdl(**e).logits
        out.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.vstack(out)

# ---------------------- teacher/student, with predictions kept ----------------------
def teacher_student(ds, model_name, threshold, teacher_epochs, student_epochs,
                    batch_size, lr, per_class_cap=None, pseudo_weight=1.0,
                    tag='ludwig', justification=''):
    """Threshold-based teacher/student self-training with a fine-tuned encoder.

    per_class_cap=None reproduces Ludwig et al.'s aggressive strategy: accept every
    unlabelled sample above the threshold. A cap reproduces the proposed method's
    conservative policy on the same backbone, isolating the labelling policy from the
    effect of fine-tuning.
    """
    activate(ds, quiet=True)
    txt = TXT
    print('\n[%s | %s] teacher on %s labelled samples'
          % (ds, tag, format(len(txt['train']), ',')))
    teacher, tok = finetune(model_name, txt['train'], Y['train'],
                            teacher_epochs, batch_size, lr, tag='%s-teacher' % tag)
    vp = predict_proba_ft(teacher, tok, txt['val'])
    val_acc = accuracy_score(Y['val'], vp.argmax(1))
    print('[%s] teacher validation accuracy = %.4f  (monitoring only)' % (tag, val_acc))

    up = predict_proba_ft(teacher, tok, txt['unl'])
    conf, pred = up.max(1), up.argmax(1)
    take = conf >= threshold
    if per_class_cap:
        capped = np.zeros_like(take)
        for c in np.unique(pred):
            ci = np.where(take & (pred == c))[0]
            if ci.size > per_class_cap: ci = ci[np.argsort(-conf[ci])[:per_class_cap]]
            capped[ci] = True
        take = capped
    purity = accuracy_score(Y['unl'][take], pred[take]) if take.sum() else np.nan
    print('[%s] accepted %s/%s (%.2f%%) at purity %.4f'
          % (tag, format(int(take.sum()), ','), format(len(take), ','),
             take.mean() * 100, purity))
    del teacher
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    stu_texts  = list(txt['train']) + list(txt['val']) + [txt['unl'][i] for i in np.where(take)[0]]
    stu_labels = np.concatenate([Y['train'], Y['val'], pred[take]])
    stu_w = np.concatenate([np.ones(len(Y['train']) + len(Y['val'])),
                            np.full(int(take.sum()), pseudo_weight)])
    print('[%s] student on %s samples' % (tag, format(len(stu_labels), ',')))
    t0 = time.time()
    student, tok2 = finetune(model_name, stu_texts, stu_labels, student_epochs,
                             batch_size, lr, weights=stu_w, tag='%s-student' % tag)
    train_s = time.time() - t0

    tp = predict_proba_ft(student, tok2, txt['test'])
    y_pred = tp.argmax(1)
    yt = np.asarray(Y['test'])
    acc = accuracy_score(yt, y_pred); f1m = f1_score(yt, y_pred, average='macro')
    lo_f, hi_f = bootstrap_ci(yt, y_pred, 'macro_f1')
    n_par = sum(p.numel() for p in student.parameters())
    res = {'Method': tag, 'Encoder': 'fine-tuned', 'Pseudo-labels': int(take.sum()),
           'Accuracy': acc, 'Macro F1': f1m, 'F1 CI low': lo_f, 'F1 CI high': hi_f,
           'AUC': roc_auc_score(yt, tp[:, 1]), 'Trainable params': n_par,
           'Train s': round(train_s, 1), 'teacher val acc': val_acc,
           'coverage': float(take.mean()), 'purity': float(purity)}
    save_preds(ds, tag, y_pred, tp)
    TEST_LEDGER.append({'#': len(TEST_LEDGER) + 1, 'Dataset': ds, 'Model': tag,
                        'Accuracy': acc, 'Macro F1': f1m})
    print('[TEST EVALUATION %d] %s/%s: accuracy %.4f, macro F1 %.4f, AUC %.4f'
          % (len(TEST_LEDGER), ds, tag, acc, f1m, res['AUC']))
    del student
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return res

print('fine-tuning utilities ready')

fine-tuning utilities ready


In [15]:
if RUN_LUDWIG:
    seed_all(42)
    historical_path=RUN_ROOT/'ludwig_bert_teacher_student.json'
    if historical_path.exists(): historical=json.loads(historical_path.read_text())
    else:
        historical=teacher_student('hatexplain',**LUDWIG,per_class_cap=None,pseudo_weight=1.,
            tag='Ludwig-style BERT teacher-student',justification='Original BERT implementation')
        atomic_json(historical_path,historical)
    rows=SIMCSE_RESULTS+[{'dataset':'HateXplain','method':historical['Method'],
        'macro_f1':historical['Macro F1'],'accuracy':historical['Accuracy'],'roc_auc':historical['AUC'],
        'pseudo_labels':historical['Pseudo-labels'],'coverage':historical['coverage'],
        'repetitions':1,'labeled_pool':'teacher L; student L+V+pseudo'}]
else:
    rows=SIMCSE_RESULTS
pd.DataFrame(rows).to_csv(RUN_ROOT/'table6_benchmarks.csv',index=False)
display(pd.DataFrame(rows))
print('Save the output folder, including predictions and partition manifests.')


[hatexplain | Ludwig-style BERT teacher-student] teacher on 1,993 labelled samples


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

... (14 outputs left out)

inference:   0%|          | 0/25 [00:00<?, ?it/s]

[TEST EVALUATION 1] hatexplain/Ludwig-style BERT teacher-student: accuracy 0.7744, macro F1 0.7743, AUC 0.8503


,dataset,method,macro_f1,accuracy,roc_auc,pseudo_labels,coverage,f1_ci_low,f1_ci_high,repetitions,ensemble_members,labeled_pool,labeled_rows,requested_rounds,completed_rounds,seconds_excluding_shared_initial_ensemble,benchmark_id
0,HateXplain,SimCSE confidence,0.728175,0.728177,0.804222,26,0.002608,0.712892,0.743219,1,5.0,L+V,2492.0,1.0,1.0,6.552698,5b9c550ab5302f6f
1,HateXplain,SimCSE full,0.721327,0.721438,0.800835,996,0.099920,0.705656,0.736753,1,5.0,L+V,2492.0,3.0,3.0,13.163382,5b9c550ab5302f6f
2,HateXplain,Ludwig-style BERT teacher-student,0.774304,0.774390,0.850292,7646,0.767055,NaN,NaN,1,NaN,teacher L; student L+V+pseudo,NaN,NaN,NaN,NaN,NaN


Save the output folder, including predictions and partition manifests.
